In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

with zipfile.ZipFile('/content/drive-download-20260612T140644Z-3-001.zip', 'r') as zf:
    zf.extractall('/content/datasets')

for root, dirs, files in os.walk('/content/datasets'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
!pip install open_clip_torch -q

In [ ]:
import numpy as np
import torch
from PIL import Image
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', device)

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(device)
model.eval()
print('CLIP ViT-B-32 loaded')

def series_to_image(x):
    length = len(x)
    side = int(np.ceil(np.sqrt(length)))
    padded = np.zeros(side * side, dtype=np.float32)
    padded[:length] = x
    grid = padded.reshape(side, side)
    grid = grid - grid.min()
    if grid.max() > 0:
        grid = grid / grid.max()
    img = (grid * 255).astype(np.uint8)
    return Image.fromarray(img).convert('RGB')

def get_tivit_embeddings(X, batch_size=32):
    n_samples, n_channels, seq_len = X.shape
    channel_embeddings = []
    with torch.no_grad():
        for c in range(n_channels):
            embs = []
            for i in range(0, n_samples, batch_size):
                batch = X[i:i+batch_size, c, :]
                imgs = torch.stack([preprocess(series_to_image(b)) for b in batch]).to(device)
                emb = model.encode_image(imgs)
                embs.append(emb.cpu().numpy())
            channel_embeddings.append(np.concatenate(embs, axis=0))
            print(f'  Channel {c+1}/{n_channels} done')
    return np.concatenate(channel_embeddings, axis=1)

In [ ]:
data = np.load('/content/datasets/CMJ.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

train_emb = get_tivit_embeddings(X_tr)
test_emb = get_tivit_embeddings(X_te)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)

acc = accuracy_score(y_te, y_pred)
print(f'TiViT Accuracy on CMJ: {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save('/content/drive/MyDrive/TiViT_CMJ_train_emb.npy', train_emb)
np.save('/content/drive/MyDrive/TiViT_CMJ_test_emb.npy', test_emb)

In [ ]:
import matplotlib.pyplot as plt

sample_idx = 0
channel_idx = 0
raw_signal = X_tr[sample_idx, channel_idx, :]
img = series_to_image(raw_signal)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(raw_signal)
axes[0].set_title('Raw time series (CMJ, channel 1)')
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Value')

axes[1].imshow(img)
axes[1].set_title('Converted to image')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/TiViT_CMJ_example_conversion.png', dpi=150)
plt.show()

In [ ]:
data = np.load('/content/datasets/MP8.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

train_emb = get_tivit_embeddings(X_tr)
test_emb = get_tivit_embeddings(X_te)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)

acc = accuracy_score(y_te, y_pred)
print(f'TiViT Accuracy on MP8: {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save('/content/drive/MyDrive/TiViT_MP8_train_emb.npy', train_emb)
np.save('/content/drive/MyDrive/TiViT_MP8_test_emb.npy', test_emb)

In [ ]:
data_mp8 = np.load('/content/datasets/MP8.npy', allow_pickle=True).item()
X_mp8 = data_mp8['train']['X']

sample_idx = 0
n_channels_to_show = min(4, X_mp8.shape[1])

fig, axes = plt.subplots(2, n_channels_to_show, figsize=(4*n_channels_to_show, 6))

for c in range(n_channels_to_show):
    raw_signal = X_mp8[sample_idx, c, :]
    img = series_to_image(raw_signal)

    axes[0, c].plot(raw_signal)
    axes[0, c].set_title(f'Channel {c+1}')
    axes[0, c].set_xlabel('Timestep')

    axes[1, c].imshow(img)
    axes[1, c].axis('off')

axes[0, 0].set_ylabel('Raw signal')
axes[1, 0].set_ylabel('Converted image')

plt.suptitle('MP8 sample: each channel converted into its own independent image')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/TiViT_MP8_example_conversion.png', dpi=150)
plt.show()

In [ ]:
data = np.load('/content/datasets/MP50.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

train_emb = get_tivit_embeddings(X_tr)
test_emb = get_tivit_embeddings(X_te)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)

acc = accuracy_score(y_te, y_pred)
print(f'TiViT Accuracy on MP50: {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save('/content/drive/MyDrive/TiViT_MP50_train_emb.npy', train_emb)
np.save('/content/drive/MyDrive/TiViT_MP50_test_emb.npy', test_emb)

In [ ]:
data = np.load('/content/datasets/SYNTH.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

train_emb = get_tivit_embeddings(X_tr)
test_emb = get_tivit_embeddings(X_te)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)

acc = accuracy_score(y_te, y_pred)
print(f'TiViT Accuracy on SYNTH: {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save('/content/drive/MyDrive/TiViT_SYNTH_train_emb.npy', train_emb)
np.save('/content/drive/MyDrive/TiViT_SYNTH_test_emb.npy', test_emb)

In [ ]:
data = np.load('/content/datasets/Rowing.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

train_emb = get_tivit_embeddings(X_tr)
test_emb = get_tivit_embeddings(X_te)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)

acc = accuracy_score(y_te, y_pred)
print(f'TiViT Accuracy on ROWING: {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save('/content/drive/MyDrive/TiViT_ROWING_train_emb.npy', train_emb)
np.save('/content/drive/MyDrive/TiViT_ROWING_test_emb.npy', test_emb)

In [ ]:
import zipfile
import glob
from google.colab import files

with zipfile.ZipFile('/content/tivit_embeddings.zip', 'w') as zf:
    for f in glob.glob('/content/drive/MyDrive/TiViT_*.npy') + glob.glob('/content/drive/MyDrive/TiViT_*.png'):
        zf.write(f, arcname=f.split('/')[-1])

files.download('/content/tivit_embeddings.zip')